# YOLOv4 Practical Notebook — 2020

Unlike v3/v5+, v4 here uses OpenCV's DNN module directly with real Darknet-format weights, since that's the standard practical way to run it (no `ultralytics` support for v4). Practical/comparison-focused, not architecture theory.

### What's new vs. the previous version

- Combined many proven techniques into one release: CSPDarknet53 backbone, Mosaic augmentation, PANet neck, CIoU loss
- CIoU loss gives a usable gradient even when boxes don't overlap at all (plain IoU loss gives zero gradient in that case)
- DIoU-NMS: post-processing that also considers center-point distance, so it doesn't wrongly delete boxes for two real objects standing close together

### Step 1: Download the pretrained Darknet weights + config

In [ ]:
from pathlib import Path
from urllib.request import urlopen

model_dir = Path("model")
model_dir.mkdir(parents=True, exist_ok=True)

files = {
    "yolov4.weights": "https://github.com/AlexeyAB/darknet/releases/download/darknet_yolo_v3_optimal/yolov4.weights",
    "yolov4.cfg": "https://raw.githubusercontent.com/AlexeyAB/darknet/master/cfg/yolov4.cfg",
    "coco.names": "https://raw.githubusercontent.com/AlexeyAB/darknet/master/data/coco.names",
}

for filename, url in files.items():
    target = model_dir / filename
    if not target.exists():
        print(f"Downloading {filename}...")
        with urlopen(url) as response, target.open("wb") as out_file:
            out_file.write(response.read())
    else:
        print(f"Already present: {target}")

### Step 2: Get the shared test image

In [ ]:
from pathlib import Path
from urllib.request import urlopen

image_path = Path("bus.jpg")
if not image_path.exists():
    print("Downloading test image...")
    with urlopen("https://ultralytics.com/images/bus.jpg") as response, image_path.open("wb") as out_file:
        out_file.write(response.read())
else:
    print("Test image already present:", image_path)

'wget' is not recognized as an internal or external command,
operable program or batch file.


### Step 3: Load the model via OpenCV's DNN module

In [ ]:
import cv2
import numpy as np

# Load class names (COCO's 80 classes)
class_labels = open("model/coco.names").read().strip().split("\n")

# Load the network directly from Darknet's .cfg (architecture) + .weights (trained params)
yolo_model = cv2.dnn.readNetFromDarknet('model/yolov4.cfg', 'model/yolov4.weights')

# Get the output layer names (YOLOv4 has 3 output scales, same idea as v3's multi-scale heads)
model_layers = yolo_model.getLayerNames()
output_indexes = yolo_model.getUnconnectedOutLayers()
output_indexes = output_indexes.flatten() if hasattr(output_indexes, 'flatten') else output_indexes
output_layers = [model_layers[int(i) - 1] for i in output_indexes]

print("Output layers:", output_layers)

FileNotFoundError: [Errno 2] No such file or directory: 'model/coco.names'

### Step 4: Run inference + time it

In [ ]:
import time
import matplotlib.pyplot as plt
from pathlib import Path

image_path = Path('bus.jpg')
test_img = cv2.imread(str(image_path))
if test_img is None:
    raise FileNotFoundError(f"Unable to read image at {image_path}")
img_height, img_width = test_img.shape[:2]

# Convert image to the blob format the network expects: normalized, resized, BGR->RGB
blob = cv2.dnn.blobFromImage(test_img, 1.0/255.0, (416, 416), swapRB=True, crop=False)
yolo_model.setInput(blob)

start = time.time()
layer_outputs = yolo_model.forward(output_layers)
elapsed = time.time() - start
print(f"Inference time: {elapsed:.3f}s")

# Parse raw detections: each output layer gives many candidate boxes, most low-confidence
boxes, confidences, class_ids = [], [], []
CONFIDENCE_THRESHOLD = 0.5

for layer_output in layer_outputs:
    for detection in layer_output:
        scores = detection[5:]
        class_id = np.argmax(scores)
        confidence = float(scores[class_id])
        if confidence > CONFIDENCE_THRESHOLD:
            box = detection[0:4] * np.array([img_width, img_height, img_width, img_height])
            (center_x, center_y, w, h) = box.astype("int")
            x = int(center_x - w / 2)
            y = int(center_y - h / 2)
            boxes.append([x, y, int(w), int(h)])
            confidences.append(confidence)
            class_ids.append(class_id)

# NMS: collapse overlapping duplicate boxes down to one per real object
nms_indices = cv2.dnn.NMSBoxes(boxes, confidences, CONFIDENCE_THRESHOLD, 0.4)
if hasattr(nms_indices, 'flatten'):
    nms_indices = nms_indices.flatten()
elif isinstance(nms_indices, list):
    nms_indices = [int(i[0]) if isinstance(i, (list, tuple, np.ndarray)) else int(i) for i in nms_indices]

result_img = test_img.copy()
print(f"Objects detected after NMS: {len(nms_indices)}")
for i in nms_indices:
    i = int(i)
    x, y, w, h = boxes[i]
    label = f"{class_labels[class_ids[i]]}: {confidences[i]:.2f}"
    print(" ", label)
    cv2.rectangle(result_img, (x, y), (x + w, y + h), (0, 255, 0), 2)
    cv2.putText(result_img, label, (x, y - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

plt.figure(figsize=(10, 8))
plt.imshow(cv2.cvtColor(result_img, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.show()

### Step 5 (optional, not run): training YOLOv4 on a custom dataset

Official v4 training uses the original Darknet C framework (`AlexeyAB/darknet`), not a simple Python `.train()` call:
```bash
# !git clone https://github.com/AlexeyAB/darknet
# !cd darknet && make
# !./darknet detector train data/obj.data cfg/yolov4-custom.cfg yolov4.conv.137
```

### One-line takeaway

v4 didn't invent new detection theory -- it systematically combined the best proven tricks (CSPDarknet53, Mosaic, PANet, CIoU) into one optimized package.